# Lab 01: Multi-Source Bronze Ingestion & Data Staging
**Course:** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Target:** 4th-Year B.Sc. Engineering Students  
**Duration:** 2 Hours (120 Minutes)  
**Session Rhythm:** Tuesday Hands-on Lab

---

## 🎯 1. Context & Objectives
In modern industrial data engineering, raw data from shopfloor SCADA networks, maintenance logs, and ERP databases arrives in heterogeneous formats with varying schemas and frequencies.

In this lab, you will establish the **Bronze Layer** of your Medallion Pipeline:
1. Locate and inspect the raw data files for your assigned industrial case study:
   * Case #1: `case1_smartpress_automotive` (CSV telemetry, JSON maintenance tickets, Excel master data)
   * Case #2: `case2_dataco_supply_chain` (CSV sales orders, JSON shipment events, Excel master catalog)
   * Case #3: `case3_nexis_workforce` (CSV shift production, JSON workforce events, Excel roster & skills)
2. Ingest CSV, JSON, and multi-sheet Excel files into `pandas` DataFrames.
3. Append standard enterprise **Audit Lineage Columns**:
   * `_ingestion_timestamp_utc` (ISO 8601 UTC timestamp)
   * `_source_file` (name of the raw source file)
4. Stage the untampered raw data into immutable **columnar Parquet files** in `data/bronze_staged/`.

> **The Golden Bronze Rule:** Never alter, filter, or clean physical measurements in the Bronze layer! Bronze is an exact, auditable replica of operational reality. Cleaning, outlier rejection, and imputation happen in the **Silver layer (Lab 02 & 03)**.


## ⚙️ 2. Environment Setup & Robust Path Resolution
First, import the required libraries and dynamically resolve the repository root and case study directories.


In [ ]:
# Step 1: Imports
import json
from datetime import datetime, timezone
from pathlib import Path
try:
    import pandas as pd
except ImportError:
    pd = None

# Step 2: Robust repository root and data discovery
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'requirements.txt').exists() and not (REPO_ROOT / '02_data_generator').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Step 3: Configure case study paths (Default: Case Study #1 SmartPress Automotive)
# (If your squad is assigned Case #2 or #3, change the folder name below accordingly!)
CASE_NAME = 'case1_smartpress_automotive'

# Check if data is located directly in lab_01/data or in 05_case_studies
LAB_DIR = Path.cwd().resolve()
if not (LAB_DIR / 'data').exists() and (REPO_ROOT / '03_labs_student_starters' / 'lab_01_bronze_ingestion' / 'data').exists():
    LAB_DIR = REPO_ROOT / '03_labs_student_starters' / 'lab_01_bronze_ingestion'

if (LAB_DIR / 'data' / CASE_NAME).exists():
    CASE_DIR = LAB_DIR / 'data' / CASE_NAME
    BRONZE_RAW = CASE_DIR / 'bronze' if (CASE_DIR / 'bronze').exists() else CASE_DIR
    BRONZE_STAGED = CASE_DIR / 'bronze_staged'
else:
    CASE_DIR = REPO_ROOT / '05_case_studies_reference_solutions' / CASE_NAME
    BRONZE_RAW = CASE_DIR / 'data' / 'bronze'
    BRONZE_STAGED = CASE_DIR / 'data' / 'bronze_staged'

# Ensure the output staging directory exists
BRONZE_STAGED.mkdir(parents=True, exist_ok=True)

print(f"Repository Root: {REPO_ROOT}")
print(f"Selected Case Study: {CASE_NAME}")
print(f"Raw Bronze Directory exists: {BRONZE_RAW.exists()}")
print(f"Staged Directory: {BRONZE_STAGED}")


## 📥 3. Part 1: Ingesting Telemetry CSV & Attaching Audit Lineage
Now, load the raw CSV file (`sensor_telemetry.csv` for SmartPress, `order_items_trans.csv` for DataCo, or `shift_production_performance.csv` for Nexis).

### Your Task:
1. Read the CSV file into a DataFrame `df_telemetry`.
2. Generate the current UTC timestamp as an ISO-8601 string: `datetime.now(timezone.utc).isoformat()`.
3. Add `_ingestion_timestamp_utc` and `_source_file` columns to the DataFrame.
4. Display the DataFrame `.shape`, column types via `.info()`, and preview the first 5 records using `.head()`.


In [ ]:
# TODO: Student Implementation - Ingest CSV
now_utc = datetime.now(timezone.utc).isoformat()
csv_file_name = 'sensor_telemetry.csv'  # Adjust if Case 2 or 3
csv_path = BRONZE_RAW / csv_file_name

# 1. Read CSV using pandas
# df_telemetry = ...

# 2. Attach audit columns
# df_telemetry['_ingestion_timestamp_utc'] = ...
# df_telemetry['_source_file'] = ...

# 3. Inspect shape and preview records
# print(f"Ingested {len(df_telemetry)} records.")
# df_telemetry.head()


## 📋 4. Part 2: Ingesting Semi-Structured JSON Logs
Next, load the JSON source (`maintenance_logs.json` for SmartPress, `shipment_events.json` for DataCo, or `workforce_events.json` for Nexis).

### Your Task:
1. Open and load the JSON file using Python's built-in `json.load()`.
2. Convert the list of JSON records into a tabular `pandas.DataFrame` named `df_maintenance`.
3. Attach the audit lineage columns (`_ingestion_timestamp_utc` and `_source_file`).
4. Display the shape and preview the first 5 maintenance tickets.


In [ ]:
# TODO: Student Implementation - Ingest JSON
json_file_name = 'maintenance_logs.json'
json_path = BRONZE_RAW / json_file_name

# 1. Load JSON file using json.load()
# with open(json_path, 'r', encoding='utf-8') as f:
#     json_data = ...

# 2. Convert to pandas DataFrame
# df_maintenance = ...

# 3. Attach audit columns
# df_maintenance['_ingestion_timestamp_utc'] = ...
# df_maintenance['_source_file'] = ...

# 4. Preview
# print(f"Ingested {len(df_maintenance)} maintenance records.")
# df_maintenance.head()


## 📊 5. Part 3: Ingesting Multi-Sheet Master Data Excel
Operational systems frequently manage enterprise master data (machine catalogs, shifts, product lists, customer segments) in multi-sheet Excel workbooks (`machines_and_shifts.xlsx` or `master_catalog.xlsx`).

### Your Task:
1. Inspect available sheets using `pd.ExcelFile`.
2. Load each sheet into its own dedicated DataFrame (e.g. `df_machines` and `df_shifts`).
3. Attach audit lineage columns to both DataFrames.
4. Preview the schemas and contents of both tables.


In [ ]:
# TODO: Student Implementation - Ingest Multi-Sheet Excel
excel_file_name = 'machines_and_shifts.xlsx'
excel_path = BRONZE_RAW / excel_file_name

# 1. Inspect sheet names using pd.ExcelFile
# excel_file = pd.ExcelFile(excel_path)
# print("Available sheets:", excel_file.sheet_names)

# 2. Read each sheet into a DataFrame
# df_machines = pd.read_excel(excel_path, sheet_name='Machines')
# df_shifts = pd.read_excel(excel_path, sheet_name='Shifts')

# 3. Attach audit columns to both DataFrames
# df_machines['_ingestion_timestamp_utc'] = now_utc
# df_machines['_source_file'] = excel_file_name
# df_shifts['_ingestion_timestamp_utc'] = now_utc
# df_shifts['_source_file'] = excel_file_name

# 4. Preview DataFrames
# display(df_machines)
# display(df_shifts)


## 💾 6. Part 4: Columnar Parquet Staging
Now that all heterogeneous sources are loaded into memory and stamped with audit metadata, stage them as **immutable Parquet files** inside `data/bronze_staged/`.

### Why Parquet?
* Unlike CSV, Parquet is a **columnar storage format** that embeds schema data types (timestamps, floats, integers) directly into file metadata.
* It uses Snappy compression, drastically reducing storage size.
* Downstream engines like **DuckDB** can query Parquet directly using zero-copy scans.

### Your Task:
1. Export `df_telemetry`, `df_maintenance`, `df_machines`, and `df_shifts` to Parquet using `.to_parquet(path, index=False)`.
2. Verify that files were created on disk and print their file sizes.


In [ ]:
# TODO: Student Implementation - Export to Parquet
# Ensure index=False so artificial pandas integer indices are not saved!

out_telem = BRONZE_STAGED / 'bronze_telemetry.parquet'
out_maint = BRONZE_STAGED / 'bronze_maintenance.parquet'
out_mach = BRONZE_STAGED / 'bronze_machines.parquet'
out_shifts = BRONZE_STAGED / 'bronze_shifts.parquet'

# df_telemetry.to_parquet(out_telem, index=False)
# df_maintenance.to_parquet(out_maint, index=False)
# df_machines.to_parquet(out_mach, index=False)
# df_shifts.to_parquet(out_shifts, index=False)

# Verification check:
# for p in [out_telem, out_maint, out_mach, out_shifts]:
#     print(f"Created: {p.name} ({p.stat().st_size / 1024:.2f} KB)")


## 🔍 7. Exploratory Inspection: Spotting Injected Anomalies
Before concluding Lab 01, take 5 minutes to explore the staged data and observe the dirty data problems that you will solve in Lab 02 and Lab 03!

Run the exploratory cell below to inspect anomalies.


In [ ]:
# Exploratory Anomaly Inspection (Preview for Lab 02 & 03)
# df_test = pd.read_parquet(out_telem)
# print("Unique Machine IDs in raw telemetry:")
# print(df_test['machine_id'].unique())
# print("\nTorque Summary Statistics (look at min and max!):")
# print(df_test['torque_Nm'].describe())
# print("\nMissing value count:")
# print(df_test.isna().sum())
